**Table of contents**<a id='toc0_'></a>    
- 1. [大模型分布式微调训练的基本概念](#toc1_)    
  - 1.1. [为什么需要分布式训练？](#toc1_1_)    
  - 1.2. [分布式训练的核心技术](#toc1_2_)    
    - 1.2.1. [数据并行（Data Parallelism）](#toc1_2_1_)    
    - 1.2.2. [模型并行（Model Parallelism）](#toc1_2_2_)    
    - 1.2.3. [流水线并行（Pipeline Parallelism）](#toc1_2_3_)    
    - 1.2.4. [混合并行（3D并行）](#toc1_2_4_)    
- 2. [deepspeed框架介绍](#toc2_)    
  - 2.1. [核心技术](#toc2_1_)    
    - 2.1.1. [ZeRO](#toc2_1_1_)    
    - 2.1.2. [ZeRO-1：](#toc2_1_2_)    
    - 2.1.3. [ZeRO-2：](#toc2_1_3_)    
    - 2.1.4. [ZeRO-3：](#toc2_1_4_)    
    - 2.1.5. [显存优化技术](#toc2_1_5_)    
  - 2.2. [优势与特点](#toc2_2_)    
  - 2.3. [使用场景](#toc2_3_)    
  - 2.4. [llamafactory上的多卡训练](#toc2_4_)    
  - 2.5. [异常问题](#toc2_5_)    
    - 2.5.1. [显存无法释放](#toc2_5_1_)    
- 3. [XTuner微调大模型](#toc3_)    
  - 3.1. [简介](#toc3_1_)    
  - 3.2. [安装](#toc3_2_)    
    - 3.2.1. [新建虚拟环境](#toc3_2_1_)    
    - 3.2.2. [克隆源码安装](#toc3_2_2_)    
    - 3.2.3. [验证安装](#toc3_2_3_)    
  - 3.3. [训练](#toc3_3_)    
    - 3.3.1. [数据集准备](#toc3_3_1_)    
      - 3.3.1.1. [自定义数据集](#toc3_3_1_1_)    
      - 3.3.1.2. [标准数据集](#toc3_3_1_2_)    
    - 3.3.2. [训练支持列表](#toc3_3_2_)    
    - 3.3.3. [训练配置文件](#toc3_3_3_)    
    - 3.3.4. [修改配置文件](#toc3_3_4_)    
      - 3.3.4.1. [1、模型位置](#toc3_3_4_1_)    
      - 3.3.4.2. [数据集位置](#toc3_3_4_2_)    
      - 3.3.4.3. [3、文本最大长度](#toc3_3_4_3_)    
      - 3.3.4.4. [批次大小](#toc3_3_4_4_)    
      - 3.3.4.5. [轮次](#toc3_3_4_5_)    
      - 3.3.4.6. [保存间隔](#toc3_3_4_6_)    
      - 3.3.4.7. [评估](#toc3_3_4_7_)    
      - 3.3.4.8. [8、修改微调方法](#toc3_3_4_8_)    
      - 3.3.4.9. [lora参数](#toc3_3_4_9_)    
      - 3.3.4.10. [修改PART 3的dataset](#toc3_3_4_10_)    
      - 3.3.4.11. [dtype](#toc3_3_4_11_)    
      - 3.3.4.12. [checkpoint](#toc3_3_4_12_)    
    - 3.3.5. [开始训练](#toc3_3_5_)    
      - 3.3.5.1. [1、单卡训练](#toc3_3_5_1_)    
      - 3.3.5.2. [2、多卡训练](#toc3_3_5_2_)    
    - 3.3.6. [训练日志](#toc3_3_6_)    
    - 3.3.7. [训练损失](#toc3_3_7_)    
    - 3.3.8. [训练权重](#toc3_3_8_)    
  - 3.4. [模型转换](#toc3_4_)    
  - 3.5. [模型合并](#toc3_5_)    
  - 3.6. [聊天模版](#toc3_6_)    
  - 3.7. [异常处理](#toc3_7_)    
    - 3.7.1. [No module named 'triton.ops'](#toc3_7_1_)    
- 4. [LLamaFactory与Xtuner多卡微调大模型](#toc4_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[大模型分布式微调训练的基本概念](#toc0_)

## 1.1. <a id='toc1_1_'></a>[为什么需要分布式训练？](#toc0_)

模型规模爆炸：

现代大模型（如GPT-3、LLaMA等）参数量达千亿级别，单卡GPU无法存储完整模型。

计算资源需求：

训练大模型需要海量计算（如GPT-3需数万GPU小时），分布式训练可加速训练过程。

内存瓶颈：

单卡显存不足以容纳大模型参数、梯度及优化器状态。

注：分布式训练目前默认指的是同型号的显卡。

## 1.2. <a id='toc1_2_'></a>[分布式训练的核心技术](#toc0_)

### 1.2.1. <a id='toc1_2_1_'></a>[数据并行（Data Parallelism）](#toc0_)

原理：  
----将数据划分为多个批次，分发到不同设备，每个设备拥有完整的模型副本。  
同步方式：  
----通过All-Reduce操作同步梯度（如PyTorch的DistributedDataParallel）。  
挑战：  
----通信开销大，显存占用高（需存储完整模型参数和优化器状态）。  
使用场景：  
----一张卡就可以跑这个模型，但是一张卡跑的太慢了，想跑的快一点

### 1.2.2. <a id='toc1_2_2_'></a>[模型并行（Model Parallelism）](#toc0_)

原理：将模型切分到不同设备（如按层或张量分片）。  
类型：  
----横向并行（层拆分）：将模型的层分配到不同设备。  
----纵向并行（张量拆分）：如Megatron-LM将矩阵乘法分片。  
挑战：设备间通信频繁，负载均衡需精细设计。 
使用场景：  
----显卡的算力或者显存不足以跑这个完整模型，但是我有多张同型号的显卡卡。


### 1.2.3. <a id='toc1_2_3_'></a>[流水线并行（Pipeline Parallelism）](#toc0_)

原理：将模型按层划分为多个阶段（stage），数据分块后按流水线执行。  
优化：微批次（Micro-batching）减少流水线气泡（Bubble）。  
挑战：需平衡阶段划分，避免资源闲置。



### 1.2.4. <a id='toc1_2_4_'></a>[混合并行（3D并行）](#toc0_)

组合策略：结合数据并行、模型并行、流水线并行，典型应用如训练千亿级模型。

案例：微软Turing-NLG、Meta的LLaMA-2。


# 2. <a id='toc2_'></a>[deepspeed框架介绍](#toc0_)

## 2.1. <a id='toc2_1_'></a>[核心技术](#toc0_)

### 2.1.1. <a id='toc2_1_1_'></a>[ZeRO](#toc0_)

ZeRO（Zero Redundancy Optimizer）

--原理：通过分片优化器状态、梯度、参数，消除数据并行中的显存冗余。  

阶段划分：

### 2.1.2. <a id='toc2_1_2_'></a>[ZeRO-1：](#toc0_)

优化器状态分片。

----每张卡上面放的都是完整的模型， 每个优化器只跟新模型的一部分，其他部分称为冗余部分，不参与计算。deepspeed框架参数的none就是ZeRO-1，只有一张卡的情况下用跟不用没有任何区别。如果只有一张卡，ZeRO-2、ZeRO-3不仅不会加速，还会减慢训练推理过程。

### 2.1.3. <a id='toc2_1_3_'></a>[ZeRO-2：](#toc0_)

梯度分片 + 优化器状态分片。 

----多张卡才可以配置

### 2.1.4. <a id='toc2_1_4_'></a>[ZeRO-3：](#toc0_)

参数分片 + 梯度分片 + 优化器状态分片。 

----多张卡才可以配置，最节约显存，但是最慢

优势：显存占用随设备数线性下降，支持训练更大模型。  

### 2.1.5. <a id='toc2_1_5_'></a>[显存优化技术](#toc0_)
梯度检查点（Activation Checkpointing）：用时间换空间，减少激活值显存占用。

CPU Offloading：将优化器状态和梯度卸载到CPU内存。

混合精度训练：FP16/BP16与动态损失缩放（Loss Scaling）。

其他特性

大规模推理支持：模型并行推理（如ZeRO-Inference）。

自适应通信优化：自动选择最佳通信策略（如All-Reduce vs. All-Gather）。


## 2.2. <a id='toc2_2_'></a>[优势与特点](#toc0_)

显存效率高：ZeRO-3可将显存占用降低至1/设备数。

易用性强：通过少量代码修改即可应用（如DeepSpeed配置JSON文件）。

扩展性优秀：支持千卡级集群训练。

开源社区支持：持续更新，与Hugging Face等生态深度集成。



## 2.3. <a id='toc2_3_'></a>[使用场景](#toc0_)

训练百亿/千亿参数模型（如GPT-3、Turing-NLG）。

资源受限环境：单机多卡训练时通过Offloading扩展模型规模。

快速实验：通过ZeRO-2加速中等规模模型训练。


## 2.4. <a id='toc2_4_'></a>[llamafactory上的多卡训练](#toc0_)

使用前要先

pip install deepSpeed

注意，这个要在服务停止的情况下装，如果服务没停的情况下装可能调不到，需要重启服务。

常规设置和以前一样

![](Image/2025-05-02-13-31-14.png)

这里会有所不同，如果服务器有两张卡，“设备数量”就会是2。DeepSpeed stage对应ZeRO-1、ZeRO-2、ZeRO-3。单张卡的时候选择none实际上不会调用DeepSpeed。

![](Image/2025-05-02-13-26-21.png)

选择none的时候两张卡都放完整的模型，不会节约显存。一般选择2，即ZeRO-2,显存占用低，速度更快。选择ZeRO-2的时候，有时显存占用看不出来区别，但是速度是更快的，可以感觉到收敛的时间比none的时候更短，且显卡的性能越强，这种差距越明显。

![](Image/2025-05-02-13-43-09.png)

一般不要勾选"使用offload",他会把部分显存中的数据放到内存中，虽然会节约显存，但是会减慢速度

<img src="./Image/2025-05-02-13-46-18.png" style="margin-left: 0" width="30%">


如果想做到极致的显存优化可以进行以下配置。但是训练速度会慢。有的时候无法使用offload，原因一般是不支持显存和内存之间的通讯或者cuda版本和deepspeed要求不匹配。

<img src="./Image/2025-05-02-14-11-12.png" style="margin-left: 0" width="80%">


## 2.5. <a id='toc2_5_'></a>[异常问题](#toc0_)

### 2.5.1. <a id='toc2_5_1_'></a>[显存无法释放](#toc0_)

有的时候已经关了服务，但是还是有一张卡的显存占用没有被释放，需要手动杀死进程。

输入top查看显存占用，找到想结束进程的PID，然后kill -9 660

<img src="./Image/2025-05-02-14-00-29.png" style="margin-left: 0" width="80%">


# 3. <a id='toc3_'></a>[XTuner微调大模型](#toc0_)

## 3.1. <a id='toc3_1_'></a>[简介](#toc0_)

[XTuner 的中文文档](https://xtuner.readthedocs.io/zh-cn/latest/)

Xtuner没有界面，但是训练速度相对llamafactory要更快。Xtuner门槛更高一点。

## 3.2. <a id='toc3_2_'></a>[安装](#toc0_)

### 3.2.1. <a id='toc3_2_1_'></a>[新建虚拟环境](#toc0_)

conda create -n envxtuner python==3.10 -y

conda activate envxtuner

### 3.2.2. <a id='toc3_2_2_'></a>[克隆源码安装](#toc0_)

新建文件夹，进入文件夹采用源码安装

cd /root/ModelFineTuningTool/Xtuner

git clone https://github.com/InternLM/xtuner.git

cd xtuner

pip install -e '.[deepspeed]'

或者可以将安装源替换

pip install -e '.[deepspeed]' -i https://pypi.tuna.tsinghua.edu.cn/simple


![](Image/2025-05-02-16-13-08.png)

### 3.2.3. <a id='toc3_2_3_'></a>[验证安装](#toc0_)

为了验证 XTuner 是否安装正确，我们将使用命令打印配置文件。

打印配置文件： 在命令行中使用 xtuner list-cfg 验证是否能打印配置文件列表。

xtuner list-cfg

<img src="./Image/2025-05-02-20-04-26.png" style="margin-left: 0" width="80%">


## 3.3. <a id='toc3_3_'></a>[训练](#toc0_)

### 3.3.1. <a id='toc3_3_1_'></a>[数据集准备](#toc0_)

附：
[xtuner中文文档](https://xtuner.readthedocs.io/zh-cn/latest/index.html)


#### 3.3.1.1. <a id='toc3_3_1_1_'></a>[自定义数据集](#toc0_)


Xtuner数据集格式如下：

![](Image/2025-05-02-16-22-54.png)

例：

将弱智吧的原始数据转换为Xtuner数据：

弱智吧原始数据：
转换后Xtuner数据：
<img src="./Image/2025-05-02-22-45-26.png" style="margin-left: 0" width="80%">


转换后Xtuner数据：

<img src="./Image/2025-05-02-22-46-12.png" style="margin-left: 0" width="80%">


数据集转换脚本data_utils.py如下：

![](Image/2025-05-02-22-50-55.png)

In [ ]:
import json

# 源数据文件路径
source_file = 'data/ruozhiba_qaswift.json'
# 目标数据文件路径
target_file = 'data/target_data.json'

# 读取源数据
with open(source_file, 'r', encoding='utf-8') as f:
    source_data = json.load(f)

# 转换数据
target_data = []
for item in source_data:
    conversation = {
        "conversation": [
            {
                "input": item["query"],
                "output": item["response"]
            }
        ]
    }
    target_data.append(conversation)

# 保存转换后的数据
with open(target_file, 'w', encoding='utf-8') as f:
    json.dump(target_data, f, ensure_ascii=False, indent=4)

print(f"数据已成功转换并保存到 {target_file}")

执行.py脚本获取target_data.json

cd /root/AI-WSL/data/xtuner数据集转换代码/data

python data_utils.py



#### 3.3.1.2. <a id='toc3_3_1_2_'></a>[标准数据集](#toc0_)

XTuner 内置了众多 map_fn （这里），可以满足大多数开源数据集的需要。此处我们罗列一些常用 map_fn 及其对应的原始字段和参考数据集：

![](Image/2025-05-05-21-28-20.png)

### 3.3.2. <a id='toc3_3_2_'></a>[训练支持列表](#toc0_)

Xtuner可以微调的前提是模型在它的支持列表中，/root/ModelFineTuningTool/Xtuner/xtuner/xtuner/configs，其中/root/ModelFineTuningTool/Xtuner是自定义的文件夹。xtuner/xtuner/configs是Xtuner自己的文件夹。Xtuner支持的模型列表如下：

<img src="./Image/2025-05-02-20-10-58.png" style="margin-left: 0" width="30%">


以qwen为例，可以看到目前只支持到1.5B版本的，模型支持没有那么全面，这也是它的缺点之一。

<img src="./Image/2025-05-02-20-15-44.png" style="margin-left: 0" width="30%">

### 3.3.3. <a id='toc3_3_3_'></a>[训练配置文件](#toc0_)

![](Image/2025-05-02-20-22-17.png)

可以看到有两个配置，qwen1_5_0_5b_chat_full_alpaca_e3.py和qwen1_5_0_5b_chat_qlora_alpaca_e3.py，其中带有full字样的是预训练用的，微调一般用不中带有full字样的。微调的时候需要修改对应的配置。

![](Image/2025-05-02-20-32-32.png)

### 3.3.4. <a id='toc3_3_4_'></a>[修改配置文件](#toc0_)

复制配置文件到xtuner目录下，例如：复制

/root/ModelFineTuningTool/Xtuner/xtuner/configs/qwen/qwen1_5/qwen1_5_0_5b_chat/qwen1_5_0_5b_chat_qlora_alpaca_e3.py

到

/root/ModelFineTuningTool/Xtuner/xtuner/qwen1_5_0_5b_chat_qlora_alpaca_e3.py

打开复制的配置文件修改如下几个地方：

#### 3.3.4.1. <a id='toc3_3_4_1_'></a>[1、模型位置](#toc0_)

pretrained_model_name_or_path = "/root/AI-WSL/models/Qwen/Qwen1___5-0___5B-Chat"

![](Image/2025-05-02-22-36-57.png)

#### 3.3.4.2. <a id='toc3_3_4_2_'></a>[数据集位置](#toc0_)

将转换后的数据集拷贝到Xtuner下的文件夹下，（不拷贝也行，反正路径要对应就行），

把alpaca_en_path = "tatsu-lab/alpaca"替换成自定义数据集名称和路径：

data_files = "/root/ModelFineTuningTool/Xtuner/xtuner/data/target_data.json"

这里的data_files是数据集的名称，可以自己定义

一、自定义的.json数据集

![](Image/2025-05-02-23-08-07.png)

二、标准数据集

标准数据集只需要传入文件夹路径就行，如果有多个数据集，都放到文件夹里面就行。但是检验就算是多个数据集也建议放到同一个文件里面，比如说有两个数据集，分别有100条数据和1000条数据，那么100条数据的那个数据随机性就会变的很差。


![](Image/2025-05-05-21-52-43.png)

![](Image/2025-05-05-21-51-23.png)

#### 3.3.4.3. <a id='toc3_3_4_3_'></a>[3、文本最大长度](#toc0_)

max_length = 512

<img src="./Image/2025-05-02-23-12-02.png" style="margin-left: 0" width="80%">

#### 3.3.4.4. <a id='toc3_3_4_4_'></a>[批次大小](#toc0_)

batch_size = 10

![](Image/2025-05-02-23-25-20.png)

#### 3.3.4.5. <a id='toc3_3_4_5_'></a>[轮次](#toc0_)

max_epochs = 1000

![](Image/2025-05-02-23-27-09.png)

#### 3.3.4.6. <a id='toc3_3_4_6_'></a>[保存间隔](#toc0_)

save_steps指的是每训练多少个批次保存一次权重。save_total_limit表示保存权重个数，2表示保存最新的两个权重。

save_steps = 50       #每隔50轮保存一次
save_total_limit = 2  # Maximum checkpoints to keep (-1 means unlimited)

![](Image/2025-05-02-23-32-24.png)

#### 3.3.4.7. <a id='toc3_3_4_7_'></a>[评估](#toc0_)

evaluation_freq要和save_steps保持一致,从训练数据集中挑选几个比较看重的数据作为验证问题。后面训练过程会输出回答，看回答和理想情况的差异来评估是否要终止训练。

evaluation_freq = 50
evaluation_inputs= ["只剩一个心脏了还能活吗？", "樟脑丸是我吃过最难吃的硬糖有奇怪的味道怎么还有人买" ,"为什么没人说ABCD型的成语？🤔","为什麽我老婆内裤拔下来没有马赛克？"]


![](Image/2025-05-02-23-40-57.png)

#### 3.3.4.8. <a id='toc3_3_4_8_'></a>[8、修改微调方法](#toc0_)

默认的load_in_4bit=True,是4位微调，如果想用8位微调就把load_in_4bit=False,load_in_8bit=True.

![](Image/2025-05-02-23-56-07.png)

如果不想做Qlora微调，只想做lora微调，则将quantization_config注释掉，不过一般用Qlora训练。

![](Image/2025-05-03-00-01-34.png)

#### 3.3.4.9. <a id='toc3_3_4_9_'></a>[lora参数](#toc0_)

lora_alpha一般是r的两倍，这里的LoRA秩默认是64，比llamafactory要大，实际要根据显存占用情况调整。


<img src="./Image/2025-05-03-00-12-13.png" style="margin-left: 0" width="60%">

#### 3.3.4.10. <a id='toc3_3_4_10_'></a>[修改PART 3的dataset](#toc0_)

如果第2步用的是：一、自定义的.json数据集  
因为我们的数据集是.json文件，所以path设置为"json"，后面data_files设置为数据集的名称。我们第2步设置的数据集名称为data_files
dataset=dict(type=load_dataset, path="json",data_files=data_files),
dataset_map_fn=None,

![](Image/2025-05-03-00-25-38.png)

如果第2步用的是：二、标准数据集

![](Image/2025-05-05-22-05-14.png)

#### 3.3.4.11. <a id='toc3_3_4_11_'></a>[dtype](#toc0_)

dtype就相当于llamafactory训练参数里面的“计算类型”，llamafactory默认用的是bf16，着重追求一些新的模型，Xtuner默认用的float16，兼容性更好些。


![](Image/2025-05-03-00-32-26.png)

#### 3.3.4.12. <a id='toc3_3_4_12_'></a>[checkpoint](#toc0_)

如果训练终止了，想在之前训练的基础上继续训练，就设置“load_from”参数，值是权重所在路径，相当于llamafactory里面的检查点。如果不需要就设置为load_from = None

load_from = "/root/ModelFineTuningTool/Xtuner/xtuner/work_dirs/qwen1_5_0_5b_chat_qlora_alpaca_e3/iter_12300.pth"

![](Image/2025-05-05-22-35-29.png)

修改完成后记得保存。

### 3.3.5. <a id='toc3_3_5_'></a>[开始训练](#toc0_)

#### 3.3.5.1. <a id='toc3_3_5_1_'></a>[1、单卡训练](#toc0_)

在当前目录下，输入以下命令启动微调脚本

xtuner train qwen1_5_0_5b_chat_qlora_alpaca_e3.py

![](Image/2025-05-05-12-08-13.png)

#### 3.3.5.2. <a id='toc3_3_5_2_'></a>[2、多卡训练](#toc0_)

多卡训练需要安装deepspeed，然后在启动时候修改两个参数，一是GPU数量，二是加速器类型。

一、GPU数量设置

![](Image/2025-05-05-16-11-33.png)

二、加速器设置

--以下命令根据需要任选其一  
xtuner train xxx --deepspeed deepspeed_zero1  
xtuner train xxx --deepspeed deepspeed_zero2  
xtuner train xxx --deepspeed deepspeed_zero2_offload  
xtuner train xxx --deepspeed deepspeed_zero3  
xtuner train xxx --deepspeed deepspeed_zero3_offload  

![](Image/2025-05-05-16-05-46.png)

综合：

例如，单机双卡：

NPROC_PER_NODE=2 xtuner train qwen1_5_0_5b_chat_qlora_alpaca_e3.py --deepspeed deepspeed_zero2

### 3.3.6. <a id='toc3_3_6_'></a>[训练日志](#toc0_)

![](Image/2025-05-05-15-30-12.png)

### 3.3.7. <a id='toc3_3_7_'></a>[训练损失](#toc0_)

![](Image/2025-05-05-15-38-45.png)

### 3.3.8. <a id='toc3_3_8_'></a>[训练权重](#toc0_)

<img src="./Image/2025-05-05-15-33-19.png" style="margin-left: 0" width="30%">

如果在多卡训练中用了deepspeed加速，这里的权重iter_9200.pth就是文件夹，如果没有deepspeed加速，就是单独的文件。这里的.pth指的是Pytorch格式的模型，一般推理部署用的huggingface格式的。

## 3.4. <a id='toc3_4_'></a>[模型转换](#toc0_)

模型训练后会自动保存成 PTH 模型（例如 iter_2000.pth ，如果使用了 DeepSpeed，则将会是一个文件夹），我们需要利用 xtuner convert pth_to_hf 将其转换为 HuggingFace 模型，以便于后续使用。具体命令为：

xtuner convert pth_to_hf ${FINETUNE_CFG} ${PTH_PATH} ${SAVE_PATH}

----FINETUNE_CFG:指的是微调模型时候用的配置文件路径，如本次案例用的qwen1_5_0_5b_chat_qlora_alpaca_e3.py，目的是获取基础模型的路径。

----PTH_PATH：指的是训练后权重路径

----SAVE_PATH：指的是转换为 HuggingFace 模型保存路径，如果文件夹不存在会自动创建。

附：
[xtuner中文文档](https://xtuner.readthedocs.io/zh-cn/latest/index.html)

修改 torch.load 参数，weights_only=False。

![](Image/2025-05-05-20-22-27.png)

否则会报错

![](Image/2025-05-05-20-27-00.png)


异常原因参考[PyTorch 2.6 默认 weights_only=True 机制详解](https://blog.csdn.net/qq_43356449/article/details/147192685)

这个错误是由于 PyTorch 2.6 引入的 默认weights_only=True 安全性增强机制导致的。当尝试加载包含非张量对象（如自定义类 HistoryBuffer）的模型权重文件时，PyTorch 默认会拒绝加载。

PyTorch 社区的安全策略升级‌  
--‌默认行为变更‌  
PyTorch 2.6 将 torch.load 的 weights_only 参数默认值从 False 改为 True，强制用户显式选择是否信任外部检查点文件24。

--开发者适配成本与安全权衡‌  
该变动虽然增加了旧代码的兼容性维护成本，但显著降低了以下场景的风险：  
----从不可信来源加载预训练模型
----分布式团队共享模型文件
----开源社区模型分发

修改后执行xtuner convert pth_to_hf ${FINETUNE_CFG} ${PTH_PATH} ${SAVE_PATH}


  相对路径写法：  
 xtuner convert pth_to_hf qwen1_5_0_5b_chat_qlora_alpaca_e3.py ./work_dirs/qwen1_5_0_5b_chat_qlora_alpaca_e3/iter_12300.pth /root/AI-WSL/models/Qwen/iter_12300_

绝对路径写法：  
 xtuner convert pth_to_hf qwen1_5_0_5b_chat_qlora_alpaca_e3.py /root/ModelFineTuningTool/Xtuner/xtuner/work_dirs/qwen1_5_0_5b_chat_qlora_alpaca_e3/iter_12300.pth /root/AI-WSL/models/Qwen/iter_12300_

![](Image/2025-05-05-20-17-46.png)

转换结果

<img src="./Image/2025-05-05-20-36-35.png" style="margin-left: 0" width="20%">

## 3.5. <a id='toc3_5_'></a>[模型合并](#toc0_)

如果使用了 LoRA / QLoRA 微调，则模型转换后将得到 adapter 参数，而并不包含原 LLM 参数。如果您期望获得合并后的模型权重（例如用于后续评测），那么可以利用 xtuner convert merge ：

xtuner convert merge ${LLM} ${LLM_ADAPTER} ${SAVE_PATH}

第一个参数：base模型路径

如果不知道，可以在导入模型目录下的adapter_config.json中"base_model_name_or_path"参数看到。

![](Image/2025-05-05-21-08-41.png)

第二个参数：训练完成转换为 HuggingFace 模型所在文件夹路径

第三个参数：合并后模型路径

例：

xtuner convert merge /root/AI-WSL/models/Qwen/Qwen1___5-0___5B-Chat /root/AI-WSL/models/Qwen/Qwen1.5-0.5B-Chat-iter_12300_ /root/AI-WSL/models/Qwen/Qwen1.5-0.5B-Chat-Xtuner-merged

合并结果

<img src="./Image/2025-05-05-21-15-08.png" style="margin-left: 0" width="70%">  
<img src="./Image/2025-05-05-21-17-38.png" style="margin-left: 0" width="25%">

## 3.6. <a id='toc3_6_'></a>[聊天模版](#toc0_)

训练完的模型如果要拿到其他推理模型进行部署使用要注意模板对齐。Xtuner模板位置在  
/xtuner/xtuner/utils/templates.py

例如千问模板，要是在vllm中部署，就要将这一段转成jinjia2的格式，在llmdeploy中部署，要转换成json格式。


<img src="./Image/2025-05-05-16-40-09.png" style="margin-left: 0" width="80%">


![](Image/2025-05-05-16-25-47.png)

    qwen_chat=dict(
        SYSTEM=("<|im_start|>system\n{system}<|im_end|>\n"),
        INSTRUCTION=("<|im_start|>user\n{input}<|im_end|>\n" "<|im_start|>assistant\n"),
        SUFFIX="<|im_end|>",
        SUFFIX_AS_EOS=True,
        SEP="\n",
        STOP_WORDS=["<|im_end|>", "<|endoftext|>"],
    )

## 3.7. <a id='toc3_7_'></a>[异常处理](#toc0_)

### 3.7.1. <a id='toc3_7_1_'></a>[No module named 'triton.ops'](#toc0_)

![](Image/2025-05-04-23-53-28.png)

# 4. <a id='toc4_'></a>[LLamaFactory与Xtuner多卡微调大模型](#toc0_)

<img src="./Image/2025-05-02-20-15-44.png" style="margin-left: 0" width="40%">